In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

candidates = [
    "seattle-weather.csv",
    "seattle-weather (2).csv",
    "/mnt/data/seattle-weather.csv",
    "/mnt/data/seattle-weather (2).csv",
]

DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Place the Seattle weather CSV in the same folder as this notebook."
    )

raw_df = pd.read_csv(DATA_PATH)
raw_df["date"] = pd.to_datetime(raw_df["date"])

print(raw_df.head())

print(f"Rows: {len(raw_df):,}")
print(f"Columns: {raw_df.shape[1]}")
print(
    f"Date range: {raw_df['date'].min().date()} "
    f"to {raw_df['date'].max().date()}"
)
print(f"Duplicate rows: {raw_df.duplicated().sum()}")

quality = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "unique": raw_df.nunique()
})

print(quality)

class_counts = raw_df["weather"].value_counts().to_frame("count")
class_counts["percent"] = (
    class_counts["count"] / len(raw_df) * 100
).round(2)

print(class_counts)

fig, ax = plt.subplots(figsize=(8, 4))

raw_df["weather"].value_counts().plot(
    kind="bar",
    ax=ax
)

ax.set_title("Weather class distribution")
ax.set_xlabel("Weather category")
ax.set_ylabel("Number of days")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

monthly = (
    raw_df
    .set_index("date")[[
        "temp_max",
        "temp_min",
        "precipitation",
        "wind"
    ]]
    .resample("MS")
    .mean()
)

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    monthly.index,
    monthly["temp_max"],
    label="Monthly mean max temp"
)

ax.plot(
    monthly.index,
    monthly["temp_min"],
    label="Monthly mean min temp"
)

ax.set_title("Seattle temperature patterns over time")
ax.set_ylabel("Temperature")
ax.legend()

plt.tight_layout()
plt.show()

classification_df = raw_df.copy()

label_encoder = LabelEncoder()

classification_df["weather_code"] = (
    label_encoder.fit_transform(
        classification_df["weather"]
    )
)

features = [
    "precipitation",
    "temp_max",
    "temp_min",
    "wind"
]

X = classification_df[features]
y = classification_df["weather_code"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print(
    "Classes:",
    dict(enumerate(label_encoder.classes_))
)

baseline_xgb = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric="mlogloss"
)

baseline_xgb.fit(
    X_train,
    y_train
)

baseline_pred = baseline_xgb.predict(
    X_test
)

print(
    "Baseline XGBoost accuracy:",
    round(
        accuracy_score(
            y_test,
            baseline_pred
        ),
        3
    )
)

print(
    "Baseline balanced accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            baseline_pred
        ),
        3
    )
)

print(
    "Baseline macro-F1:",
    round(
        f1_score(
            y_test,
            baseline_pred,
            average="macro",
            zero_division=0
        ),
        3
    )
)

print()

print(
    classification_report(
        y_test,
        baseline_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

def evaluate_model(
    name,
    model,
    X_test,
    y_test
):
    pred = model.predict(X_test)

    return {
        "model": name,
        "accuracy": accuracy_score(
            y_test,
            pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            pred
        ),
        "macro_f1": f1_score(
            y_test,
            pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_test,
            pred,
            average="weighted",
            zero_division=0
        ),
    }

models = {}
results = []

dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(
    X_train,
    y_train
)

models["Dummy baseline"] = dummy

results.append(
    evaluate_model(
        "Dummy baseline",
        dummy,
        X_test,
        y_test
    )
)

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

rf.fit(
    X_train,
    y_train
)

models["Random Forest"] = rf

results.append(
    evaluate_model(
        "Random Forest",
        rf,
        X_test,
        y_test
    )
)

weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    eval_metric="mlogloss"
)

xgb.fit(
    X_train,
    y_train,
    sample_weight=weights
)

models["Weighted XGBoost"] = xgb

results.append(
    evaluate_model(
        "Weighted XGBoost",
        xgb,
        X_test,
        y_test
    )
)

comparison = (
    pd.DataFrame(results)
    .set_index("model")
    .sort_values(
        "macro_f1",
        ascending=False
    )
)

print(
    comparison.round(3)
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.1],
    "gamma": [0, 1],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

tuned_xgb = GridSearchCV(
    estimator=XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="mlogloss"
    ),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=0
)

tuned_xgb.fit(
    X_train,
    y_train,
    sample_weight=weights
)

print(
    "Best parameters:",
    tuned_xgb.best_params_
)

print(
    "Best CV macro-F1:",
    round(
        tuned_xgb.best_score_,
        3
    )
)

tuned_pred = tuned_xgb.predict(
    X_test
)

print(
    "Test accuracy:",
    round(
        accuracy_score(
            y_test,
            tuned_pred
        ),
        3
    )
)

print(
    "Test balanced accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            tuned_pred
        ),
        3
    )
)

print(
    "Test macro-F1:",
    round(
        f1_score(
            y_test,
            tuned_pred,
            average="macro",
            zero_division=0
        ),
        3
    )
)

print()

print(
    classification_report(
        y_test,
        tuned_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    tuned_pred,
    display_labels=label_encoder.classes_,
    cmap="Blues",
    xticks_rotation=45,
    ax=ax
)

ax.set_title(
    "Tuned XGBoost confusion matrix"
)

plt.tight_layout()
plt.show()

best_xgb = tuned_xgb.best_estimator_

importance = pd.Series(
    best_xgb.feature_importances_,
    index=features
).sort_values()

fig, ax = plt.subplots(
    figsize=(7, 4)
)

importance.plot(
    kind="barh",
    ax=ax
)

ax.set_title(
    "XGBoost feature importance"
)

ax.set_xlabel(
    "Importance"
)

plt.tight_layout()
plt.show()

print(
    importance
    .sort_values(
        ascending=False
    )
    .to_frame("importance")
)

forecast_df = (
    raw_df
    .sort_values("date")
    .copy()
)

forecast_df["target_weather"] = (
    forecast_df["weather"].shift(-1)
)

base_features = [
    "precipitation",
    "temp_max",
    "temp_min",
    "wind"
]

for lag in [1, 2, 3]:
    for col in base_features:
        forecast_df[
            f"{col}_lag{lag}"
        ] = forecast_df[col].shift(lag)

for col in base_features:
    forecast_df[
        f"{col}_roll3_mean"
    ] = (
        forecast_df[col]
        .shift(1)
        .rolling(3)
        .mean()
    )

next_day = (
    forecast_df["date"]
    + pd.Timedelta(days=1)
)

forecast_df[
    "next_month"
] = next_day.dt.month

forecast_df[
    "next_dayofyear"
] = next_day.dt.dayofyear

forecast_df = pd.get_dummies(
    forecast_df,
    columns=["weather"],
    prefix="today_weather",
    dtype=int
)

forecast_df = (
    forecast_df
    .dropna()
    .reset_index(drop=True)
)

forecast_encoder = LabelEncoder()

forecast_df["target_code"] = (
    forecast_encoder.fit_transform(
        forecast_df["target_weather"]
    )
)

forecast_features = [
    c
    for c in forecast_df.columns
    if "_lag" in c
    or "_roll3_mean" in c
]

forecast_features += [
    "next_month",
    "next_dayofyear"
]

forecast_features += [
    c
    for c in forecast_df.columns
    if c.startswith("today_weather_")
]

split_idx = int(
    len(forecast_df) * 0.8
)

train_f = forecast_df.iloc[
    :split_idx
]

test_f = forecast_df.iloc[
    split_idx:
]

X_train_f = train_f[
    forecast_features
]

y_train_f = train_f[
    "target_code"
]

X_test_f = test_f[
    forecast_features
]

y_test_f = test_f[
    "target_code"
]

print(
    "Forecast training period:",
    train_f["date"].min().date(),
    "to",
    train_f["date"].max().date()
)

print(
    "Forecast test period:",
    test_f["date"].min().date(),
    "to",
    test_f["date"].max().date()
)

print(
    "Forecast feature count:",
    len(forecast_features)
)

forecast_results = []

dummy_f = DummyClassifier(
    strategy="most_frequent"
)

dummy_f.fit(
    X_train_f,
    y_train_f
)

forecast_results.append(
    evaluate_model(
        "Dummy baseline",
        dummy_f,
        X_test_f,
        y_test_f
    )
)

rf_f = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    min_samples_leaf=2,
    random_state=RANDOM_STATE
)

rf_f.fit(
    X_train_f,
    y_train_f
)

forecast_results.append(
    evaluate_model(
        "Random Forest",
        rf_f,
        X_test_f,
        y_test_f
    )
)

forecast_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_f
)

xgb_f = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    eval_metric="mlogloss"
)

xgb_f.fit(
    X_train_f,
    y_train_f,
    sample_weight=forecast_weights
)

forecast_results.append(
    evaluate_model(
        "Weighted XGBoost",
        xgb_f,
        X_test_f,
        y_test_f
    )
)

forecast_comparison = (
    pd.DataFrame(
        forecast_results
    )
    .set_index("model")
    .sort_values(
        "macro_f1",
        ascending=False
    )
)

print(
    forecast_comparison.round(3)
)

best_forecast_name = (
    forecast_comparison.index[0]
)

best_forecast_model = {
    "Dummy baseline": dummy_f,
    "Random Forest": rf_f,
    "Weighted XGBoost": xgb_f
}[best_forecast_name]

forecast_pred = (
    best_forecast_model.predict(
        X_test_f
    )
)

print(
    "Best next-day model:",
    best_forecast_name
)

all_forecast_labels = np.arange(
    len(
        forecast_encoder.classes_
    )
)

print(
    classification_report(
        y_test_f,
        forecast_pred,
        labels=all_forecast_labels,
        target_names=forecast_encoder.classes_,
        zero_division=0
    )
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

ConfusionMatrixDisplay.from_predictions(
    y_test_f,
    forecast_pred,
    labels=all_forecast_labels,
    display_labels=forecast_encoder.classes_,
    cmap="Blues",
    xticks_rotation=45,
    ax=ax
)

ax.set_title(
    "Next-day weather prediction confusion matrix"
)

plt.tight_layout()
plt.show()
